# NB06 — Cross-Method Comparison
**MSK | Goel Lab** · LLM Systematic Review · Method Comparison

**Purpose**: Compute and compare evaluation metrics (precision, recall, F1, thematic coverage, reproducibility, time-to-results, screening burden) across all three methods against the gold standard reference set.

**Inputs**
- `data/splits/gold_standard_set.csv` — manually curated reference set
- `experiments/*.json` — run records per method
- `data/processed/extracted_features_*.csv`

**Outputs**
- `data/processed/method_comparison.csv` — all metrics in one table
- `reports/tables/comparison_table.csv`

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.io import load_env, data_dir, project_root, experiments_dir, load_gold_standard
load_env()

import json
import pandas as pd
from pathlib import Path

## 1. Load gold standard + run records

In [ ]:
gold_ids = load_gold_standard()
print(f'Gold standard: {len(gold_ids)} articles')

# Load most recent run per method
run_files = sorted(experiments_dir().glob('*.json'))
runs_by_method = {}
for f in run_files:
    r = json.loads(f.read_text())
    m = r.get('method', 'unknown')
    runs_by_method.setdefault(m, []).append(r)

# Use latest run per method
latest = {m: sorted(rs, key=lambda x: x.get('run_id',''))[-1]
          for m, rs in runs_by_method.items()}
print('Methods with runs:', list(latest.keys()))

## 2. Compute retrieval metrics per method

In [ ]:
from src.evaluation.metrics import compute_retrieval_metrics, metrics_to_dataframe

metrics_list = []
for method, run in latest.items():
    m = compute_retrieval_metrics(
        retrieved_ids=run.get('retrieved_doc_ids', []),
        gold_ids=list(gold_ids),
        method=method,
        run_id=run.get('run_id'),
        time_seconds=run.get('time_seconds'),
        screening_burden=run.get('n_deduped') or run.get('n_retrieved'),
    )
    metrics_list.append(m)

comparison_df = metrics_to_dataframe(metrics_list)
comparison_df

## 3. Thematic coverage

In [ ]:
from src.evaluation.metrics import compute_thematic_coverage
import yaml

theme_kw_path = project_root() / 'eval' / 'schemas' / 'theme_keywords.yaml'
if theme_kw_path.exists():
    with open(theme_kw_path) as f:
        theme_keywords = yaml.safe_load(f)

    for method, run in latest.items():
        doc_ids = run.get('retrieved_doc_ids', [])
        corpus_df = pd.read_csv(data_dir() / 'corpus' / 'corpus_metadata.csv')
        texts = corpus_df[corpus_df['doc_id'].isin(doc_ids)]['abstract'].dropna().tolist()
        cov = compute_thematic_coverage(texts, theme_keywords)
        frac = sum(cov.values()) / len(cov)
        print(f'{method}: thematic coverage = {frac:.2f} ({sum(cov.values())}/{len(cov)} themes)')
else:
    print('theme_keywords.yaml not yet created — skipping thematic coverage.')

## 4. Save comparison table

In [ ]:
comparison_df.to_csv(data_dir() / 'processed' / 'method_comparison.csv', index=False)
comparison_df.to_csv(project_root() / 'reports' / 'tables' / 'comparison_table.csv', index=False)
print('Saved comparison table.')
comparison_df